# Polymarket WNBA Trading Backtest — Convergence Exit

Walk-forward protocol: train XGB on 2015-2024 (early-stop on 2024), predict 2025.  
sigma_p = ensemble std dev (20 bootstrap models).

**Backtest mechanics**
- Entry decision: single snapshot at tipoff - 5 minutes
- Executable prices: `mid + HALF_SPREAD` to buy YES (home token), `(1 - mid) + HALF_SPREAD` to buy NO (away token)
- Exit: sell position when market odds converge to model prediction
  - YES position: exit when `mid - HALF_SPREAD >= p_xgb`
  - NO position: exit when `mid + HALF_SPREAD <= p_xgb` (i.e. away bid >= 1 - p_xgb)
- Monitoring: every 1 minute after tipoff until convergence or settlement
- Cost: HALF_SPREAD = 0.02 (2c) applied to every entry and every convergence exit;
  captures realistic execution slippage vs mid-price data
- One entry per market per strategy; if both sides pass, take larger net edge

**Why 2c half-spread?**  
Polymarket's `/prices-history` returns 1-min mid prices (no BBO).  
WNBA moneyline markets have lower liquidity than major-sport lines.  
Tick size is 1c; a 2c half-spread (4c round-trip) is conservative but realistic.

**Policies tested**

| Group | Policy | Criterion |
|---|---|---|
| Edge-only | E1 | \|edge\| >= 0.03 |
| Edge-only | E2 | \|edge\| >= 0.05 |
| Edge-only | E3 | \|edge\| >= 0.08 |
| Edge-only | E4 | \|edge\| >= 0.12 |
| Edge+sigma | S1 | \|edge\| >= 0.05, sigma <= 0.05 |
| Edge+sigma | S2 | \|edge\| >= 0.08, sigma <= 0.05 |
| Edge+sigma | S3 | \|edge\| >= 0.08, sigma <= 0.04 |
| Ratio | Z2 | \|edge\| >= 0.05, z >= 1.5 |

z = |edge| / (sigma_p + 0.01)

In [8]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT_ROOT = Path('..').resolve()
GOLD_DIR     = PROJECT_ROOT / 'data' / 'gold'
POLY_DIR     = PROJECT_ROOT / 'data' / 'polymarket'

# --- XGB params: tuning3 winner (confirmed by follow-up grid) ---
XGB_PARAMS = dict(
    objective='binary:logistic', eval_metric='logloss',
    max_depth=6, min_child_weight=2, subsample=0.9,
    colsample_bytree=0.8, reg_lambda=0.5, reg_alpha=0.0,
    gamma=0.5, learning_rate=0.03, seed=42, nthread=-1,
)
N_PLAYERS  = 7
MAX_ROUNDS = 3000
EARLY_STOP = 150
N_ENS      = 20
ENS_SEED   = 42
HALF_SPREAD = 0.02      # 2c half-spread applied to mid prices
CLIP_EPS   = 1e-7
LABEL_COL  = 'home_win'

# --- Feature columns (same as 32_run_xgboost_cv.py) ---
PLAYER_FEATS = ['m_ewma_pre','q_pre','days_since_first_report_pre','days_since_last_dnp_pre',
                'consec_dnps_pre','played_last_game_pre','minutes_last_game_pre',
                'days_since_last_played_pre','injury_present_flag_pre']
FORM_FEATS   = ['net_rtg_ewma_pre','efg_ewma_pre','tov_pct_ewma_pre','orb_pct_ewma_pre','ftr_ewma_pre']
STYLE_FEATS  = ['off_3pa_rate_pre','def_3pa_allowed_pre','off_2pa_rate_pre','def_2pa_allowed_pre',
                'off_tov_pct_pre','def_forced_tov_pre']
SCHED_FEATS  = ['days_rest_pre','is_b2b_pre','games_last_4_days_pre','games_last_7_days_pre',
                'travel_miles_pre','timezone_shift_hours_pre']

def build_feature_cols(n):
    cols = [f'{s}_p{k}_{f}' for s in ('home','away') for k in range(1, n+1) for f in PLAYER_FEATS]
    for f in FORM_FEATS + STYLE_FEATS + SCHED_FEATS:
        cols += [f'home_{f}', f'away_{f}']
    return cols

FEAT_COLS = build_feature_cols(N_PLAYERS)

def clip(p): return np.clip(p, CLIP_EPS, 1 - CLIP_EPS)

def load_year(year):
    df = pd.read_csv(GOLD_DIR / f'game_xgboost_input_{year}_REGPST.csv')
    cold = (df['home_p1_m_ewma_pre'] == 0) | (df['away_p1_m_ewma_pre'] == 0)
    return df[~cold].reset_index(drop=True)

def make_dm(df, use_bm=True):
    avail = [c for c in FEAT_COLS if c in df.columns]
    y = df[LABEL_COL].values.astype(float) if LABEL_COL in df.columns else None
    dm = xgb.DMatrix(df[avail].values.astype(float), label=y,
                     feature_names=avail, missing=np.nan)
    if use_bm and 'base_margin' in df.columns:
        dm.set_base_margin(df['base_margin'].values.astype(float))
    return dm

print(f'Feature cols: {len(FEAT_COLS)}')
print(f'HALF_SPREAD: {HALF_SPREAD} (round-trip: {2*HALF_SPREAD})')
print('Setup complete.')

Feature cols: 160
HALF_SPREAD: 0.02 (round-trip: 0.04)
Setup complete.


## 2. Train XGB + Ensemble on 2015-2024, predict 2025

Produces `p_xgb` (single model) and `sigma_p` (ensemble std) for every 2025 game.

In [9]:
# Load all gold data
all_data = {}
for yr in range(2015, 2026):
    all_data[yr] = load_year(yr)
    print(f'  {yr}: {len(all_data[yr])} rows')

test_2025    = all_data[2025]
es_tr        = pd.concat([all_data[yr] for yr in range(2015, 2024)], ignore_index=True)
val_df       = all_data[2024]
full_tr      = pd.concat([all_data[yr] for yr in range(2015, 2025)], ignore_index=True)

# --- Step 1: early-stop on 2024 to find best_round ---
m_es = xgb.train(
    XGB_PARAMS, make_dm(es_tr), MAX_ROUNDS,
    evals=[(make_dm(val_df), 'val')],
    early_stopping_rounds=EARLY_STOP, verbose_eval=False,
)
best_round = m_es.best_iteration + 1
print(f'\nES best_round: {best_round}')

# --- Step 2: retrain on 2015-2024, predict 2025 ---
m_final = xgb.train(XGB_PARAMS, make_dm(full_tr), best_round, verbose_eval=False)
p_xgb_25 = clip(m_final.predict(make_dm(test_2025)))

# --- Step 3: ensemble (20 bootstrap models) for sigma_p ---
print(f'Training ensemble: {N_ENS} models, {best_round} rounds each...')
ens_preds = []
for i in range(N_ENS):
    seed = ENS_SEED + i
    rng  = np.random.default_rng(seed)
    idx  = rng.integers(0, len(full_tr), size=len(full_tr))
    dm_b = make_dm(full_tr.iloc[idx].reset_index(drop=True))
    m_b  = xgb.train({**XGB_PARAMS, 'seed': seed}, dm_b, best_round, verbose_eval=False)
    ens_preds.append(clip(m_b.predict(make_dm(test_2025))))

ens_stack  = np.stack(ens_preds)
p_ens_mean = ens_stack.mean(0)
sigma_p    = ens_stack.std(0)

# --- Build game-level signal table ---
signals = test_2025[['game_id', 'game_ts', 'game_date',
                      'home_team_id', 'away_team_id', LABEL_COL]].copy()
signals['p_xgb']   = p_xgb_25
signals['sigma_p']  = sigma_p
signals['game_ts']  = pd.to_datetime(signals['game_ts'], utc=True)

from sklearn.metrics import log_loss
y25 = test_2025[LABEL_COL].values.astype(float)
print(f'\nXGB  2025 log-loss: {log_loss(y25, p_xgb_25):.5f}')
print(f'Ens  2025 log-loss: {log_loss(y25, p_ens_mean):.5f}')
print(f'sigma_p: mean={sigma_p.mean():.4f}  median={np.median(sigma_p):.4f}  max={sigma_p.max():.4f}')
print(f'Games: {len(signals)}')

  2015: 218 rows
  2016: 220 rows
  2017: 219 rows
  2018: 221 rows
  2019: 220 rows
  2020: 147 rows
  2021: 209 rows
  2022: 239 rows
  2023: 260 rows
  2024: 262 rows
  2025: 310 rows

ES best_round: 17
Training ensemble: 20 models, 17 rounds each...

XGB  2025 log-loss: 0.61080
Ens  2025 log-loss: 0.61117
sigma_p: mean=0.0314  median=0.0315  max=0.0505
Games: 310


## 3. Load Polymarket data and build home-team token mapping

Each Polymarket moneyline market has two outcome tokens (Team A / Team B).  
We identify which token corresponds to the home team so the home-token mid price
equals the implied home-win probability.

**Note:** The matched file has game_id collisions (multiple games sharing one ID),
so we match on team-pair + game_date instead.

In [10]:
# --- Load Polymarket matched games ---
p_matched = pd.read_csv(POLY_DIR / 'wnba_2025_game_markets_matched.csv')

# --- Build robust match key: sorted team-pair + game_date ---
# (game_id in the matched file has collisions; team-pair is unique per date)
def team_pair_key(row, id_a_col, id_b_col, date_col):
    ids = sorted([str(row[id_a_col]), str(row[id_b_col])])
    return f"{ids[0]}_{ids[1]}_{row[date_col]}"

p_matched['match_key'] = p_matched.apply(
    lambda r: team_pair_key(r, 'team_a_id', 'team_b_id', 'game_date'), axis=1
)
signals['match_key'] = signals.apply(
    lambda r: team_pair_key(r, 'home_team_id', 'away_team_id', 'game_date'), axis=1
)

# --- Join on match_key ---
game_market = p_matched.merge(
    signals[['match_key', 'game_id', 'home_team_id', 'away_team_id',
             'game_ts', 'p_xgb', 'sigma_p', LABEL_COL]],
    on='match_key', how='inner', suffixes=('_poly', ''),
)
print(f'Matched markets (inner join): {len(game_market)}')

# --- Determine home-team token ---
game_market['home_token_id'] = np.where(
    game_market['team_a_id'] == game_market['home_team_id'],
    game_market['team_a_token_id'],
    game_market['team_b_token_id'],
)

# --- Settlement check using winner column (city names, matches team_a/team_b) ---
game_market['home_team_name'] = np.where(
    game_market['team_a_id'] == game_market['home_team_id'],
    game_market['team_a'],
    game_market['team_b'],
)
game_market['home_won_settle'] = (
    game_market['winner'] == game_market['home_team_name']
).astype(int)

# Deduplicate on match_key (not game_id, which has collisions)
home_tokens = game_market.drop_duplicates('match_key').copy()

# Drop games without a winner (unresolved)
home_tokens = home_tokens[home_tokens['winner'].notna()].copy()

settle_check = (home_tokens['home_won_settle'] == home_tokens[LABEL_COL])
print(f'Games with Polymarket home-team token: {len(home_tokens)}')
print(f'Settlement check: home_won_settle matches home_win: '
      f'{settle_check.sum()}/{len(settle_check)} '
      f'({settle_check.mean():.1%})')
if not settle_check.all():
    n_mismatch = (~settle_check).sum()
    print(f'  ({n_mismatch} mismatches — dropped)')
    home_tokens = home_tokens[settle_check].copy()
    print(f'  Remaining after drop: {len(home_tokens)}')

Matched markets (inner join): 277
Games with Polymarket home-team token: 277
Settlement check: home_won_settle matches home_win: 277/277 (100.0%)


## 4. Build entry snapshots and post-tipoff price series

- **Entry**: last 1-min mid price at or before tipoff - 5 min  
  → simulated ask = `mid + HALF_SPREAD`, simulated NO cost = `(1 - mid) + HALF_SPREAD`
- **Exit monitoring**: all 1-min mid prices after tipoff (used to detect convergence)

In [11]:
# Load price history for home-team tokens
home_token_set = set(home_tokens['home_token_id'])
prices = pd.read_csv(POLY_DIR / 'polymarket_prices_history.csv')
prices['ts'] = pd.to_datetime(prices['ts'], utc=True)
prices = prices[prices['token_id'].isin(home_token_set)].copy()
prices = prices.sort_values(['token_id', 'ts']).reset_index(drop=True)
print(f'Home-team price points loaded: {len(prices):,}')

# Build lookup: home_token_id -> game info
token_info = home_tokens.set_index('home_token_id')[
    ['game_id', 'game_ts', 'p_xgb', 'sigma_p', LABEL_COL]
].to_dict('index')

# --- Entry snapshots: last mid price at or before (tipoff - 5 min) ---
entry_rows = []
for token_id, info in token_info.items():
    tipoff = info['game_ts']
    entry_cutoff = tipoff - pd.Timedelta(minutes=5)

    sub = prices[(prices['token_id'] == token_id) & (prices['ts'] <= entry_cutoff)]
    if sub.empty:
        continue

    last = sub.iloc[-1]  # already sorted by ts
    mid = last['price']
    if not (0 < mid < 1):
        continue

    # Simulated executable prices with half-spread
    q_yes = mid + HALF_SPREAD           # cost to buy home token
    q_no  = (1 - mid) + HALF_SPREAD     # cost to buy away token

    # Skip if execution price is out of range
    if q_yes >= 1.0 or q_no >= 1.0:
        continue

    entry_rows.append({
        'game_id':        info['game_id'],
        'home_token_id':  token_id,
        'entry_ts':       last['ts'],
        'mid_entry':      mid,
        'q_yes_exec':     q_yes,
        'q_no_exec':      q_no,
        'p_xgb':          info['p_xgb'],
        'sigma_p':        info['sigma_p'],
        'home_win':       info[LABEL_COL],
        'tipoff':         tipoff,
    })

entries = pd.DataFrame(entry_rows)
print(f'Games with valid entry snapshot: {len(entries)}')
print(f'Entry times: median {(entries["tipoff"] - entries["entry_ts"]).median()} before tipoff')
print(f'Entry mid: mean={entries["mid_entry"].mean():.3f}  '
      f'q_yes_exec mean={entries["q_yes_exec"].mean():.3f}  '
      f'q_no_exec mean={entries["q_no_exec"].mean():.3f}')

# --- Post-tipoff price series (for exit monitoring) ---
post_tipoff = {}
for token_id, info in token_info.items():
    tipoff = info['game_ts']
    sub = prices[(prices['token_id'] == token_id) & (prices['ts'] > tipoff)].copy()
    if not sub.empty:
        post_tipoff[token_id] = sub[['ts', 'price']].reset_index(drop=True)

print(f'Games with post-tipoff prices: {len(post_tipoff)}')

Home-team price points loaded: 696,096
Games with valid entry snapshot: 277
Entry times: median 0 days 00:05:53 before tipoff
Entry mid: mean=0.548  q_yes_exec mean=0.568  q_no_exec mean=0.472
Games with post-tipoff prices: 277


## 5. Trading engine — convergence exit

1. At tipoff - 5 min, check entry criteria for each policy
2. Post-tipoff, scan 1-min mid prices for convergence:
   - YES position: exit when `mid - HALF_SPREAD >= p_xgb` → sell home token at simulated bid
   - NO position: exit when `mid + HALF_SPREAD <= p_xgb` → sell away token at simulated bid
3. If convergence never occurs, hold to settlement

In [12]:
# --- Policy definitions ---
def make_edge_policy(min_edge):
    def policy(ey, en, sigma):
        return max(ey, en) >= min_edge
    return policy

def make_edge_sigma_policy(min_edge, max_sigma):
    def policy(ey, en, sigma):
        return sigma <= max_sigma and max(ey, en) >= min_edge
    return policy

def make_z_policy(min_edge, min_z):
    def policy(ey, en, sigma):
        best_edge = max(ey, en)
        return best_edge >= min_edge and best_edge / (sigma + 0.01) >= min_z
    return policy

POLICIES = {
    'E1': make_edge_policy(0.03),
    'E2': make_edge_policy(0.05),
    'E3': make_edge_policy(0.08),
    'E4': make_edge_policy(0.12),
    'S1': make_edge_sigma_policy(0.05, 0.05),
    'S2': make_edge_sigma_policy(0.08, 0.05),
    'S3': make_edge_sigma_policy(0.08, 0.04),
    'Z2': make_z_policy(0.05, 1.5),
}

# --- Trading engine with convergence exit ---
def run_backtest(entries_df, post_tipoff_prices, policies, half_spread=HALF_SPREAD):
    all_trades = {name: [] for name in policies}

    for _, row in entries_df.iterrows():
        p       = row['p_xgb']
        sigma   = row['sigma_p']
        q_yes   = row['q_yes_exec']
        q_no    = row['q_no_exec']
        hw      = row['home_win']
        token   = row['home_token_id']
        game_id = row['game_id']
        tipoff  = row['tipoff']

        edge_yes = p - q_yes
        edge_no  = (1 - p) - q_no

        for pname, pfunc in policies.items():
            if not pfunc(edge_yes, edge_no, sigma):
                continue

            # Determine side
            if edge_yes >= edge_no:
                side     = 'YES'
                entry_px = q_yes
                edge     = edge_yes
            else:
                side     = 'NO'
                entry_px = q_no
                edge     = edge_no

            # --- Search for convergence exit in post-tipoff prices ---
            exit_type = 'settlement'
            exit_px   = None
            exit_ts   = None
            mins_held = None

            if token in post_tipoff_prices:
                pt = post_tipoff_prices[token]
                for _, c in pt.iterrows():
                    mid = c['price']
                    if not (0 < mid < 1):
                        continue

                    if side == 'YES' and (mid - half_spread) >= p:
                        # Simulated home-token bid reached model price
                        exit_type = 'converged'
                        exit_px   = mid - half_spread
                        exit_ts   = c['ts']
                        mins_held = (c['ts'] - tipoff).total_seconds() / 60
                        break
                    elif side == 'NO' and (mid + half_spread) <= p:
                        # Simulated away-token bid reached model price
                        exit_type = 'converged'
                        exit_px   = (1 - mid) - half_spread
                        exit_ts   = c['ts']
                        mins_held = (c['ts'] - tipoff).total_seconds() / 60
                        break

            # --- Compute PnL ---
            if exit_type == 'converged':
                # Spread cost baked into both entry_px and exit_px
                pnl = exit_px - entry_px
            else:
                # Hold to settlement: payout is exact, only entry spread cost
                payout = 1.0 if (side == 'YES' and hw == 1) or (side == 'NO' and hw == 0) else 0.0
                exit_px = payout
                pnl = payout - entry_px

            all_trades[pname].append({
                'game_id':     game_id,
                'side':        side,
                'entry_px':    entry_px,
                'edge':        edge,
                'exit_type':   exit_type,
                'exit_px':     exit_px,
                'pnl':         pnl,
                'p_xgb':       p,
                'sigma_p':     sigma,
                'home_win':    hw,
                'mins_held':   mins_held,
            })

    return {k: pd.DataFrame(v) for k, v in all_trades.items()}


trades = run_backtest(entries, post_tipoff, POLICIES)

for pname in POLICIES:
    tdf = trades[pname]
    if tdf.empty:
        print(f'{pname}:   0 trades')
        continue
    n_conv = (tdf['exit_type'] == 'converged').sum()
    n_sett = (tdf['exit_type'] == 'settlement').sum()
    print(f'{pname}: {len(tdf):3d} trades | {n_conv:3d} converged, {n_sett:3d} settlement')

E1: 192 trades | 159 converged,  33 settlement
E2: 157 trades | 129 converged,  28 settlement
E3: 110 trades |  88 converged,  22 settlement
E4:  65 trades |  48 converged,  17 settlement
S1: 157 trades | 129 converged,  28 settlement
S2: 110 trades |  88 converged,  22 settlement
S3:  82 trades |  67 converged,  15 settlement
Z2: 134 trades | 110 converged,  24 settlement


## 6. Results

In [13]:
from sklearn.metrics import brier_score_loss

summary_rows = []

for pname in POLICIES:
    tdf = trades[pname]
    if tdf.empty:
        summary_rows.append({'policy': pname, 'n_trades': 0})
        continue

    n         = len(tdf)
    n_conv    = (tdf['exit_type'] == 'converged').sum()
    n_sett    = (tdf['exit_type'] == 'settlement').sum()
    hit       = (tdf['pnl'] > 0).mean()
    mean_e    = tdf['edge'].mean()
    mean_pnl  = tdf['pnl'].mean()
    total_pnl = tdf['pnl'].sum()
    total_risk = tdf['entry_px'].sum()
    roi       = total_pnl / total_risk if total_risk > 0 else 0

    # Brier: model prob vs actual outcome on the side we traded
    brier_p = np.where(tdf['side'] == 'YES', tdf['p_xgb'], 1 - tdf['p_xgb'])
    brier_y = np.where(tdf['side'] == 'YES', tdf['home_win'], 1 - tdf['home_win'])
    brier_val = brier_score_loss(brier_y, brier_p)

    avg_sigma = tdf['sigma_p'].mean()

    conv = tdf[tdf['exit_type'] == 'converged']
    avg_mins_held = conv['mins_held'].mean() if len(conv) > 0 else np.nan

    summary_rows.append({
        'policy':       pname,
        'n_trades':     n,
        'n_converged':  n_conv,
        'n_settlement': n_sett,
        'mean_edge':    round(mean_e, 4),
        'hit_rate':     round(hit, 4),
        'mean_pnl':     round(mean_pnl, 4),
        'total_pnl':    round(total_pnl, 2),
        'roi':          round(roi, 4),
        'brier':        round(brier_val, 5),
        'avg_sigma_p':  round(avg_sigma, 4),
        'avg_mins_held': round(avg_mins_held, 1) if not np.isnan(avg_mins_held) else None,
    })

summary = pd.DataFrame(summary_rows)

pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:.4f}'.format)
print('=' * 140)
print(f'POLYMARKET TRADING BACKTEST — Convergence Exit  (2025 season, HALF_SPREAD={HALF_SPREAD})')
print('=' * 140)
print(summary.to_string(index=False))

# Save
out_dir = POLY_DIR
summary.to_csv(out_dir / 'backtest_convergence_summary.csv', index=False)
print(f'\nSaved: {out_dir}/backtest_convergence_summary.csv')

# --- Per-policy detail ---
print('\n' + '=' * 140)
for pname in POLICIES:
    tdf = trades[pname]
    if tdf.empty:
        continue
    conv = tdf[tdf['exit_type'] == 'converged']
    sett = tdf[tdf['exit_type'] == 'settlement']
    print(f'\n--- {pname}: {len(tdf)} trades | '
          f'converged={len(conv)} (PnL={conv["pnl"].sum():.2f}) | '
          f'settlement={len(sett)} (PnL={sett["pnl"].sum():.2f}) | '
          f'total PnL={tdf["pnl"].sum():.2f} ---')

    show_cols = ['entry_px', 'edge', 'exit_px', 'pnl', 'sigma_p']
    print('  All trades:')
    print(tdf[show_cols].describe().round(4).to_string())
    if len(conv) > 0:
        print(f'  Converged exits: median hold = {conv["mins_held"].median():.0f} min, '
              f'mean hold = {conv["mins_held"].mean():.0f} min')
    tdf.to_csv(out_dir / f'backtest_convergence_trades_{pname}.csv', index=False)

POLYMARKET TRADING BACKTEST — Convergence Exit  (2025 season, HALF_SPREAD=0.02)
policy  n_trades  n_converged  n_settlement  mean_edge  hit_rate  mean_pnl  total_pnl    roi  brier  avg_sigma_p  avg_mins_held
    E1       192          159            33     0.1095    0.8281    0.0840    16.1300 0.2373 0.2232       0.0328        26.3000
    E2       157          129            28     0.1253    0.8217    0.0904    14.1800 0.2609 0.2347       0.0333        29.0000
    E3       110           88            22     0.1513    0.8000    0.0984    10.8200 0.2828 0.2421       0.0343        30.7000
    E4        65           48            17     0.1883    0.7385    0.0929     6.0400 0.2824 0.2576       0.0348        39.3000
    S1       157          129            28     0.1253    0.8217    0.0904    14.1800 0.2609 0.2347       0.0333        29.0000
    S2       110           88            22     0.1513    0.8000    0.0984    10.8200 0.2828 0.2421       0.0343        30.7000
    S3        82        

## 7. Hold-to-settlement comparison

Same entry logic (tipoff - 5 min, same policies), but always hold to settlement instead of convergence exit.

In [14]:
# --- Hold-to-settlement backtest ---
def run_backtest_settle(entries_df, policies):
    all_trades = {name: [] for name in policies}

    for _, row in entries_df.iterrows():
        p       = row['p_xgb']
        sigma   = row['sigma_p']
        q_yes   = row['q_yes_exec']
        q_no    = row['q_no_exec']
        hw      = row['home_win']
        game_id = row['game_id']

        edge_yes = p - q_yes
        edge_no  = (1 - p) - q_no

        for pname, pfunc in policies.items():
            if not pfunc(edge_yes, edge_no, sigma):
                continue

            if edge_yes >= edge_no:
                side     = 'YES'
                entry_px = q_yes
                edge     = edge_yes
                payout   = 1.0 if hw == 1 else 0.0
            else:
                side     = 'NO'
                entry_px = q_no
                edge     = edge_no
                payout   = 1.0 if hw == 0 else 0.0

            pnl = payout - entry_px   # only entry-side spread cost

            all_trades[pname].append({
                'game_id':   game_id,
                'side':      side,
                'entry_px':  entry_px,
                'edge':      edge,
                'pnl':       pnl,
                'payout':    payout,
                'p_xgb':     p,
                'sigma_p':   sigma,
                'home_win':  hw,
            })

    return {k: pd.DataFrame(v) for k, v in all_trades.items()}


trades_settle = run_backtest_settle(entries, POLICIES)

# --- Summary table ---
settle_rows = []
for pname in POLICIES:
    tdf = trades_settle[pname]
    if tdf.empty:
        settle_rows.append({'policy': pname, 'n_trades': 0})
        continue

    n         = len(tdf)
    hit       = (tdf['payout'] == 1.0).mean()
    mean_e    = tdf['edge'].mean()
    mean_pnl  = tdf['pnl'].mean()
    total_pnl = tdf['pnl'].sum()
    total_risk = tdf['entry_px'].sum()
    roi       = total_pnl / total_risk if total_risk > 0 else 0

    brier_p   = np.where(tdf['side'] == 'YES', tdf['p_xgb'], 1 - tdf['p_xgb'])
    brier_y   = tdf['payout'].values
    brier_val = brier_score_loss(brier_y, brier_p)
    avg_sigma = tdf['sigma_p'].mean()

    settle_rows.append({
        'policy':      pname,
        'n_trades':    n,
        'mean_edge':   round(mean_e, 4),
        'hit_rate':    round(hit, 4),
        'mean_pnl':    round(mean_pnl, 4),
        'total_pnl':   round(total_pnl, 2),
        'roi':         round(roi, 4),
        'brier':       round(brier_val, 5),
        'avg_sigma_p': round(avg_sigma, 4),
    })

settle_summary = pd.DataFrame(settle_rows)

print('=' * 120)
print(f'POLYMARKET TRADING BACKTEST — Hold to Settlement  (2025 season, HALF_SPREAD={HALF_SPREAD})')
print('=' * 120)
print(settle_summary.to_string(index=False))

settle_summary.to_csv(out_dir / 'backtest_settlement_summary.csv', index=False)
print(f'\nSaved: {out_dir}/backtest_settlement_summary.csv')

# --- Side-by-side comparison ---
print('\n' + '=' * 120)
print('CONVERGENCE EXIT vs HOLD-TO-SETTLEMENT')
print('=' * 120)
comp_rows = []
for pname in POLICIES:
    c = summary[summary['policy'] == pname].iloc[0]
    s = settle_summary[settle_summary['policy'] == pname].iloc[0]
    comp_rows.append({
        'policy':           pname,
        'n_trades':         int(c['n_trades']),
        'conv_total_pnl':   c['total_pnl'],
        'conv_roi':         c['roi'],
        'conv_hit':         c['hit_rate'],
        'sett_total_pnl':   s['total_pnl'],
        'sett_roi':         s['roi'],
        'sett_hit':         s['hit_rate'],
        'pnl_diff':         round(c['total_pnl'] - s['total_pnl'], 2),
    })
comp_df = pd.DataFrame(comp_rows)
print(comp_df.to_string(index=False))

POLYMARKET TRADING BACKTEST — Hold to Settlement  (2025 season, HALF_SPREAD=0.02)
policy  n_trades  mean_edge  hit_rate  mean_pnl  total_pnl     roi  brier  avg_sigma_p
    E1       192     0.1095    0.3698    0.0158     3.0300  0.0446 0.2232       0.0328
    E2       157     0.1253    0.3885    0.0423     6.6400  0.1220 0.2347       0.0333
    E3       110     0.1513    0.3727    0.0249     2.7400  0.0716 0.2421       0.0343
    E4        65     0.1883    0.4000    0.0709     4.6100  0.2155 0.2576       0.0348
    S1       157     0.1253    0.3885    0.0423     6.6400  0.1220 0.2347       0.0333
    S2       110     0.1513    0.3727    0.0249     2.7400  0.0716 0.2421       0.0343
    S3        82     0.1499    0.3293   -0.0224    -1.8400 -0.0636 0.2357       0.0311
    Z2       134     0.1365    0.3955    0.0481     6.4500  0.1386 0.2323       0.0327

Saved: C:\Users\arius\Desktop\kalshi_wnba_bot\data\polymarket/backtest_settlement_summary.csv

CONVERGENCE EXIT vs HOLD-TO-SETTLEMENT
